# ALTO v0 — COCO Dataset Preparation + Multi-context Baseline

Notebook này thực hiện:

1. Mount Google Drive.
2. Tải COCO 2017 Validation và annotation nếu chưa có.
3. Tự động chọn 6 ảnh thuộc ba loại context:
   - `free_placement`
   - `object_relative`
   - `physical_interaction`
4. Tự động tạo target human-region mask.
5. Tạo `dataset_manifest.csv`.
6. Chạy SDXL Inpainting với nhiều seed trên toàn bộ 6 sample.
7. Tạo contact sheet, phiếu đánh giá và Oracle Best-of-N.

\[
x_{j,i}=G_\theta(z_{T,j}^{(i)},p_j,I_{b,j},M_j)
\]

## Cấu trúc Drive

```text
/content/drive/MyDrive/Colab Notebook/alto/
├── alto_v0.ipynb
├── Input/
│   ├── coco/
│   ├── backgrounds/
│   ├── masks/
│   └── dataset_manifest.csv
└── Output/
    ├── images/
    ├── contact_sheets/
    └── ...
```

## 1. Mount Drive và cài đặt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q -U pycocotools diffusers transformers accelerate safetensors pandas matplotlib

In [ ]:
import gc, json, math, random, shutil, time, urllib.request, zipfile
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageFilter
from IPython.display import display
from pycocotools.coco import COCO
from diffusers import AutoPipelineForInpainting

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not enabled")

## 2. Cấu hình

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebook/alto")
INPUT_DIR = PROJECT_DIR / "Input"
OUTPUT_DIR = PROJECT_DIR / "Output"

COCO_DIR = INPUT_DIR / "coco"
COCO_IMAGES_DIR = COCO_DIR / "val2017"
COCO_ANNOTATIONS_DIR = COCO_DIR / "annotations"
COCO_INSTANCES_PATH = COCO_ANNOTATIONS_DIR / "instances_val2017.json"

BACKGROUND_DIR = INPUT_DIR / "backgrounds"
MASK_DIR = INPUT_DIR / "masks"
DATASET_MANIFEST_PATH = INPUT_DIR / "dataset_manifest.csv"

for p in [
    INPUT_DIR, OUTPUT_DIR, COCO_DIR, BACKGROUND_DIR, MASK_DIR,
    OUTPUT_DIR / "images", OUTPUT_DIR / "contact_sheets"
]:
    p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "experiment_name": "alto_v0",
    "dataset_seed": 42,
    "num_seeds_per_sample": 8,
    "seed_start": 1000,
    "model_id": "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
    "width": 1024,
    "height": 1024,
    "num_inference_steps": 30,
    "guidance_scale": 7.5,
    "strength": 0.99,
    "mask_blur_radius": 8,
}

random.seed(CONFIG["dataset_seed"])
np.random.seed(CONFIG["dataset_seed"])
print(json.dumps(CONFIG, indent=2))

## 3. Tải COCO 2017 Validation và annotations

In [ ]:
COCO_VAL_URL = "http://images.cocodataset.org/zips/val2017.zip"
COCO_ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"

def download(url, path):
    path = Path(path)
    if path.exists():
        print("Đã tồn tại:", path)
        return
    print("Đang tải:", url)
    urllib.request.urlretrieve(url, path)

def extract(path, destination):
    with zipfile.ZipFile(path, "r") as z:
        z.extractall(destination)

val_zip = COCO_DIR / "val2017.zip"
ann_zip = COCO_DIR / "annotations_trainval2017.zip"

if not COCO_IMAGES_DIR.exists():
    download(COCO_VAL_URL, val_zip)
    extract(val_zip, COCO_DIR)

if not COCO_INSTANCES_PATH.exists():
    download(COCO_ANN_URL, ann_zip)
    extract(ann_zip, COCO_DIR)

assert COCO_IMAGES_DIR.exists()
assert COCO_INSTANCES_PATH.exists()

## 4. Load COCO và category IDs

In [ ]:
coco = COCO(str(COCO_INSTANCES_PATH))
cat_name_to_id = {c["name"]: c["id"] for c in coco.loadCats(coco.getCatIds())}

for name in ["person", "dining table", "car", "bench", "couch"]:
    print(name, "->", cat_name_to_id[name])

## 5. Hàm hình học, occupancy và tạo mask

In [ ]:
def xywh_to_xyxy(b):
    x, y, w, h = b
    return (x, y, x+w, y+h)

def clip_box(box, W, H):
    x1,y1,x2,y2 = box
    return (
        max(0,min(W,int(x1))), max(0,min(H,int(y1))),
        max(0,min(W,int(x2))), max(0,min(H,int(y2)))
    )

def box_area(b):
    x1,y1,x2,y2 = b
    return max(0,x2-x1)*max(0,y2-y1)

def intersection_area(a,b):
    ax1,ay1,ax2,ay2=a
    bx1,by1,bx2,by2=b
    return max(0,min(ax2,bx2)-max(ax1,bx1))*max(0,min(ay2,by2)-max(ay1,by1))

def anns_for_image(image_id):
    return coco.loadAnns(coco.getAnnIds(imgIds=[image_id]))

def has_person(image_id):
    return bool(coco.getAnnIds(
        imgIds=[image_id],
        catIds=[cat_name_to_id["person"]]
    ))

def occupancy(image_id, region):
    area = box_area(region)
    if area <= 0:
        return 1.0
    covered = sum(
        intersection_area(region, xywh_to_xyxy(a["bbox"]))
        for a in anns_for_image(image_id)
    )
    return min(1.0, covered / area)

def create_mask(size, box):
    W,H = size
    box = clip_box(box,W,H)
    mask = Image.new("L",(W,H),0)
    ImageDraw.Draw(mask).rectangle(box,fill=255)
    return mask, box

def free_box(W,H,relation):
    if relation=="left_side":
        return (0.05*W,0.10*H,0.45*W,0.98*H)
    return (0.30*W,0.18*H,0.70*W,1.00*H)

def beside_box(bbox,W,H):
    x,y,w,h=bbox
    pw=max(0.55*w,0.18*W)
    ph=max(2.0*h,0.65*H)
    if W-(x+w) >= x:
        x1,x2=x+w,x+w+pw
        side="right"
    else:
        x1,x2=x-pw,x
        side="left"
    cy=y+0.5*h
    return clip_box((x1,cy-0.55*ph,x2,cy+0.45*ph),W,H),side

def interaction_box(bbox,W,H):
    x,y,w,h=bbox
    return clip_box((x-0.35*w,y-1.45*h,x+1.35*w,y+1.10*h),W,H)

## 6. Định nghĩa 6 sample context

In [ ]:
SPECS = [
    {
        "sample_id":"sample_001",
        "context_type":"free_placement",
        "relation":"left_side",
        "reference_object":"",
        "prompt":"A photorealistic woman standing naturally in the empty space on the left side of the scene, with correct scale and perspective. Preserve all unmasked background details."
    },
    {
        "sample_id":"sample_002",
        "context_type":"free_placement",
        "relation":"center_foreground",
        "reference_object":"",
        "prompt":"A photorealistic man standing naturally in the center foreground, with scale and perspective consistent with the scene. Preserve all unmasked background details."
    },
    {
        "sample_id":"sample_003",
        "context_type":"object_relative",
        "relation":"beside",
        "reference_object":"dining table",
        "prompt":"A photorealistic woman standing naturally beside the dining table, with correct spacing, scale, and perspective. Preserve all unmasked background details."
    },
    {
        "sample_id":"sample_004",
        "context_type":"object_relative",
        "relation":"near",
        "reference_object":"car",
        "prompt":"A photorealistic man standing naturally near the car, with height, scale, and perspective consistent with the vehicle and scene. Preserve all unmasked background details."
    },
    {
        "sample_id":"sample_005",
        "context_type":"physical_interaction",
        "relation":"sitting_on",
        "reference_object":"bench",
        "prompt":"A photorealistic woman sitting naturally on the bench, with physically plausible contact, proportions, scale, and perspective. Preserve all unmasked background details."
    },
    {
        "sample_id":"sample_006",
        "context_type":"physical_interaction",
        "relation":"sitting_on",
        "reference_object":"couch",
        "prompt":"A photorealistic man sitting naturally on the couch, with physically plausible contact, proportions, scale, and perspective. Preserve all unmasked background details."
    },
]
pd.DataFrame(SPECS)

## 7. Tự động lọc và chọn ảnh COCO

In [ ]:
def choose_free(relation, excluded):
    candidates=[]
    for image_id,info in coco.imgs.items():
        if image_id in excluded or has_person(image_id):
            continue
        W,H=info["width"],info["height"]
        if W<500 or H<400:
            continue
        region=free_box(W,H,relation)
        occ=occupancy(image_id,region)
        if occ<=0.22:
            score=(1-occ)+0.15*(min(W,H)/max(W,H))
            candidates.append((score,image_id,region,occ))
    if not candidates:
        raise RuntimeError("Không tìm được free-placement candidate")
    return sorted(candidates,key=lambda x:(-x[0],x[1]))[0]

def choose_object(category,mode,excluded):
    cat_id=cat_name_to_id[category]
    candidates=[]
    for image_id in sorted(coco.getImgIds(catIds=[cat_id])):
        if image_id in excluded or has_person(image_id):
            continue
        info=coco.loadImgs([image_id])[0]
        W,H=info["width"],info["height"]
        anns=coco.loadAnns(coco.getAnnIds(
            imgIds=[image_id],catIds=[cat_id],iscrowd=False
        ))
        for ann in anns:
            x,y,w,h=ann["bbox"]
            ratio=(w*h)/(W*H)
            min_ratio=0.05 if mode=="relative" else 0.07
            if not(min_ratio<=ratio<=0.45):
                continue
            if mode=="relative":
                target,side=beside_box(ann["bbox"],W,H)
                occ=occupancy(image_id,target)
                if occ>0.38:
                    continue
                score=ratio+0.35*(1-occ)
            else:
                target=interaction_box(ann["bbox"],W,H)
                side="overlap"
                score=ratio
            candidates.append({
                "score":score,"image_id":image_id,
                "annotation_id":ann["id"],"bbox":ann["bbox"],
                "target_box":target,"placement_side":side
            })
    if not candidates:
        raise RuntimeError(f"Không tìm được candidate cho {category}")
    return sorted(candidates,key=lambda x:(-x["score"],x["image_id"],x["annotation_id"]))[0]

selected=[]
excluded=set()

for spec in SPECS:
    if spec["context_type"]=="free_placement":
        score,image_id,target_box,occ=choose_free(spec["relation"],excluded)
        row={
            **spec,"coco_image_id":image_id,"annotation_id":"",
            "reference_bbox":"","target_box":target_box,
            "selection_score":score,"region_occupancy":occ,
            "placement_side":spec["relation"],
            "mask_rule":"fixed_free_region"
        }
    else:
        mode="relative" if spec["context_type"]=="object_relative" else "interaction"
        r=choose_object(spec["reference_object"],mode,excluded)
        row={
            **spec,"coco_image_id":r["image_id"],
            "annotation_id":r["annotation_id"],
            "reference_bbox":r["bbox"],
            "target_box":r["target_box"],
            "selection_score":r["score"],
            "region_occupancy":"",
            "placement_side":r["placement_side"],
            "mask_rule":"beside_object" if mode=="relative" else "interaction_expansion"
        }
    selected.append(row)
    excluded.add(row["coco_image_id"])

selected_df=pd.DataFrame(selected)
display(selected_df)

## 8. Copy background, tạo mask và dataset manifest

In [ ]:
manifest=[]

for row in selected:
    info=coco.loadImgs([row["coco_image_id"]])[0]
    src=COCO_IMAGES_DIR/info["file_name"]
    bg_path=BACKGROUND_DIR/f'{row["sample_id"]}.jpg'
    mask_path=MASK_DIR/f'{row["sample_id"]}.png'

    shutil.copy2(src,bg_path)
    image=Image.open(bg_path).convert("RGB")
    mask,clipped=create_mask(image.size,row["target_box"])
    mask.save(mask_path)

    manifest.append({
        "sample_id":row["sample_id"],
        "coco_image_id":row["coco_image_id"],
        "annotation_id":row["annotation_id"],
        "context_type":row["context_type"],
        "relation":row["relation"],
        "reference_object":row["reference_object"],
        "reference_bbox":json.dumps(row["reference_bbox"]),
        "target_box":json.dumps(clipped),
        "mask_rule":row["mask_rule"],
        "placement_side":row["placement_side"],
        "selection_score":row["selection_score"],
        "region_occupancy":row["region_occupancy"],
        "background_path":str(bg_path.relative_to(INPUT_DIR)),
        "mask_path":str(mask_path.relative_to(INPUT_DIR)),
        "prompt":row["prompt"],
    })

dataset_df=pd.DataFrame(manifest)
dataset_df.to_csv(DATASET_MANIFEST_PATH,index=False)
display(dataset_df)
print("Saved:",DATASET_MANIFEST_PATH)

## 9. Preview 6 background và mask

In [ ]:
for row in dataset_df.itertuples(index=False):
    bg=Image.open(INPUT_DIR/row.background_path).convert("RGB")
    mask=Image.open(INPUT_DIR/row.mask_path).convert("L")

    bg.thumbnail((360,360))
    mask_rgb=mask.convert("RGB")
    mask_rgb.thumbnail((360,360))

    canvas=Image.new("RGB",(720,410),"white")
    canvas.paste(bg,(0,0))
    canvas.paste(mask_rgb,(360,0))
    draw=ImageDraw.Draw(canvas)
    draw.text((10,370),f"{row.sample_id} | {row.context_type} | COCO {row.coco_image_id}",fill="black")
    display(canvas)

## 10. Load SDXL Inpainting

In [ ]:
assert torch.cuda.is_available(), "Hãy bật GPU."

torch.cuda.empty_cache()
gc.collect()

pipe=AutoPipelineForInpainting.from_pretrained(
    CONFIG["model_id"],
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
pipe.enable_model_cpu_offload()

NEGATIVE_PROMPT=(
    "deformed body, malformed hands, extra fingers, extra limbs, "
    "floating body, incorrect perspective, distorted furniture, "
    "damaged background, duplicate person, low quality, blurry"
)

## 11. Chạy toàn bộ 6 sample với nhiều seed

In [ ]:
seeds=list(range(
    CONFIG["seed_start"],
    CONFIG["seed_start"]+CONFIG["num_seeds_per_sample"]
))
records=[]

for _,row in dataset_df.iterrows():
    sample_dir=OUTPUT_DIR/"images"/row["sample_id"]
    sample_dir.mkdir(parents=True,exist_ok=True)

    background=Image.open(INPUT_DIR/row["background_path"]).convert("RGB").resize(
        (CONFIG["width"],CONFIG["height"]),Image.Resampling.LANCZOS
    )
    mask=Image.open(INPUT_DIR/row["mask_path"]).convert("L").resize(
        (CONFIG["width"],CONFIG["height"]),Image.Resampling.NEAREST
    )
    if CONFIG["mask_blur_radius"]>0:
        mask=mask.filter(ImageFilter.GaussianBlur(CONFIG["mask_blur_radius"]))

    for seed in seeds:
        out_path=sample_dir/f"seed_{seed}.png"
        rec={
            "sample_id":row["sample_id"],
            "coco_image_id":row["coco_image_id"],
            "context_type":row["context_type"],
            "relation":row["relation"],
            "reference_object":row["reference_object"],
            "seed":seed,
            "status":"started",
            "elapsed_seconds":None,
            "prompt":row["prompt"],
            "output_path":str(out_path),
            "error":""
        }
        try:
            g=torch.Generator(device="cuda").manual_seed(seed)
            start=time.time()
            image=pipe(
                prompt=row["prompt"],
                negative_prompt=NEGATIVE_PROMPT,
                image=background,
                mask_image=mask,
                generator=g,
                num_inference_steps=CONFIG["num_inference_steps"],
                guidance_scale=CONFIG["guidance_scale"],
                strength=CONFIG["strength"],
                width=CONFIG["width"],
                height=CONFIG["height"],
            ).images[0]
            rec["elapsed_seconds"]=time.time()-start
            rec["status"]="success"
            image.save(out_path)
            print(row["sample_id"],seed,f'{rec["elapsed_seconds"]:.1f}s')
        except Exception as e:
            rec["status"]="failed"
            rec["error"]=repr(e)
            print("FAILED",row["sample_id"],seed,e)
        records.append(rec)
        torch.cuda.empty_cache()
        gc.collect()

runs_df=pd.DataFrame(records)
runs_df.to_csv(OUTPUT_DIR/"runs.csv",index=False)
display(runs_df.head())

## 12. Contact sheet theo sample

In [ ]:
def contact_sheet(group,sample_id,cols=4,thumb=256):
    group=group[group["status"]=="success"]
    if group.empty:
        return None
    label_h=34
    rows=math.ceil(len(group)/cols)
    sheet=Image.new("RGB",(cols*thumb,rows*(thumb+label_h)),"white")
    draw=ImageDraw.Draw(sheet)
    for i,row in enumerate(group.itertuples(index=False)):
        img=Image.open(row.output_path).convert("RGB")
        img.thumbnail((thumb,thumb))
        x=(i%cols)*thumb
        y=(i//cols)*(thumb+label_h)
        cell=Image.new("RGB",(thumb,thumb),"white")
        cell.paste(img,((thumb-img.width)//2,(thumb-img.height)//2))
        sheet.paste(cell,(x,y))
        draw.text((x+8,y+thumb+8),f"seed={row.seed}",fill="black")
    path=OUTPUT_DIR/"contact_sheets"/f"{sample_id}.png"
    sheet.save(path)
    return path

for sample_id,group in runs_df.groupby("sample_id"):
    p=contact_sheet(group,sample_id)
    if p:
        display(Image.open(p))

## 13. Phiếu đánh giá thủ công

In [ ]:
success_df=runs_df[runs_df["status"]=="success"].copy()
eval_df=success_df[
    ["sample_id","coco_image_id","context_type","relation",
     "reference_object","seed","prompt","output_path"]
].copy()

for col in [
    "human_presence","placement_score","scale_score",
    "relation_score","interaction_score","background_score",
    "prompt_score","artifact_score","overall_score",
    "failure_label","notes"
]:
    eval_df[col]=""

eval_df.to_csv(OUTPUT_DIR/"manual_evaluation.csv",index=False)
display(eval_df.head())

## 14. Oracle Best-of-N

In [ ]:
df=pd.read_csv(OUTPUT_DIR/"manual_evaluation.csv")
numeric=[
    "placement_score","scale_score","relation_score",
    "interaction_score","background_score","prompt_score",
    "artifact_score","overall_score"
]
for c in numeric:
    df[c]=pd.to_numeric(df[c],errors="coerce")

if df["overall_score"].isna().all():
    raise ValueError("Hãy điền manual_evaluation.csv trước.")

df["interaction_effective"]=df["interaction_score"].fillna(df["relation_score"])
df["manual_total_score"]=(
    0.15*df["placement_score"]
    +0.15*df["scale_score"]
    +0.20*df["relation_score"]
    +0.15*df["interaction_effective"]
    +0.15*df["background_score"]
    +0.10*df["prompt_score"]
    +0.20*df["overall_score"]
    -0.10*df["artifact_score"]
)

ranking=df.sort_values(
    ["sample_id","manual_total_score"],
    ascending=[True,False]
)
ranking.to_csv(OUTPUT_DIR/"manual_ranking.csv",index=False)

oracle=ranking.groupby("sample_id",as_index=False).first()
oracle.to_csv(OUTPUT_DIR/"oracle_best_of_n.csv",index=False)
display(oracle[["sample_id","context_type","seed","manual_total_score"]])

## 15. Tổng hợp theo context và lưu manifest

In [ ]:
summary=(
    df.groupby("context_type")
    .agg(
        num_candidates=("seed","count"),
        mean_score=("manual_total_score","mean"),
        std_score=("manual_total_score","std"),
        min_score=("manual_total_score","min"),
        max_score=("manual_total_score","max"),
    )
    .reset_index()
)
summary.to_csv(OUTPUT_DIR/"summary_by_context.csv",index=False)
display(summary)

experiment_manifest={
    **CONFIG,
    "created_at":datetime.now().isoformat(),
    "dataset_manifest_path":str(DATASET_MANIFEST_PATH),
    "coco_image_ids":dataset_df["coco_image_id"].tolist(),
    "seeds":seeds,
}
(OUTPUT_DIR/"experiment_manifest.json").write_text(
    json.dumps(experiment_manifest,indent=2,ensure_ascii=False),
    encoding="utf-8"
)

# Kết quả đầu ra

Notebook sẽ tạo:

```text
Input/
├── coco/
├── backgrounds/
├── masks/
└── dataset_manifest.csv

Output/
├── images/
├── contact_sheets/
├── runs.csv
├── manual_evaluation.csv
├── manual_ranking.csv
├── oracle_best_of_n.csv
├── summary_by_context.csv
└── experiment_manifest.json
```